# 04 -- Hypothesentests: H1 und H2

**Person:** A | **Input:** `data/processed/bank_segments.csv`, `data/processed/ga4_segments.csv`

**H1:** Personalisierte (segment-spezifische) Ansprache erzielt signifikant hoehere Response-Raten als generische Massenkommunikation.

**H2:** Verhaltensbasierte Segmentierung verbessert die Treffsicherheit von Zielgruppenansprache messbar (Silhouette + Conversion-Lift).

**Methoden:** Chi-Quadrat-Test (H1), Z-Test fuer Proportionen (H1 paarweise), Silhouette-Score + Conversion-Rate-Vergleich (H2)

In [1]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, norm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../outputs', exist_ok=True)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
ALPHA = 0.05
print(f'Setup OK | Signifikanzniveau: alpha={ALPHA}')

Setup OK | Signifikanzniveau: alpha=0.05


## 1. Daten laden

In [2]:
bank = pd.read_csv('../data/processed/bank_segments.csv')
ga4  = pd.read_csv('../data/processed/ga4_segments.csv')

print('Bank Marketing:')
print(bank['cluster_label'].value_counts())
print(f'Overall Response-Rate: {bank["y_binary"].mean()*100:.2f}%')
print()
print('GA4 Segmente:')
print(ga4['cluster_label'].value_counts())
print(f'Overall Conversion-Rate: {ga4["has_purchased"].mean()*100:.2f}%')

Bank Marketing:
cluster_label
Mass Market    41832
Experienced     3379
Name: count, dtype: int64
Overall Response-Rate: 11.70%

GA4 Segmente:
cluster_label
Passive         63865
Loyal Buyers     3463
Champions         491
Name: count, dtype: int64
Overall Conversion-Rate: 0.40%


## 2. H1 -- Chi-Quadrat-Test: Segment-Zugehoerigkeit vs. Response

**Nullhypothese (H0):** Die Response-Rate ist unabhaengig vom Kundensegment (kein Unterschied).

**Alternativhypothese (H1):** Mindestens ein Segment zeigt eine signifikant abweichende Response-Rate.

In [3]:
# Kontingenzmatrix: Segment x Response
contingency = pd.crosstab(bank['cluster_label'], bank['y_binary'],
                           rownames=['Segment'], colnames=['Response (0=no, 1=yes)'])
print('Kontingenzmatrix:')
print(contingency)
print()

# Chi-Quadrat-Test
chi2, p_value, dof, expected = chi2_contingency(contingency)

# Cramers V als Effektstaerke
n = contingency.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print(f'Chi-Quadrat (chi2):  {chi2:.4f}')
print(f'Freiheitsgrade (df): {dof}')
print(f'p-Wert:              {p_value:.6f}')
print(f"Cramers V:           {cramers_v:.4f}  ({'schwach' if cramers_v < 0.1 else 'mittel' if cramers_v < 0.3 else 'stark'})")
print()
print(f'Entscheidung (alpha={ALPHA}):')
if p_value < ALPHA:
    print(f'  H0 ABGELEHNT -- p={p_value:.6f} < {ALPHA}')
    print('  -> H1 BESTAETIGT: Segment-Zugehoerigkeit hat signifikanten Einfluss auf Response-Rate')
else:
    print(f'  H0 NICHT ABGELEHNT -- p={p_value:.6f} >= {ALPHA}')
    print('  -> H1 NICHT BESTAETIGT')

Kontingenzmatrix:
Response (0=no, 1=yes)      0     1
Segment                            
Experienced              2513   866
Mass Market             37409  4423

Chi-Quadrat (chi2):  684.5896
Freiheitsgrade (df): 1
p-Wert:              0.000000
Cramers V:           0.1231  (mittel)

Entscheidung (alpha=0.05):
  H0 ABGELEHNT -- p=0.000000 < 0.05
  -> H1 BESTAETIGT: Segment-Zugehoerigkeit hat signifikanten Einfluss auf Response-Rate


## 3. H1 -- Paarweiser Z-Test: Experienced vs. Mass Market

In [4]:
# Paarweiser Vergleich: bestes Segment vs. Gesamt-Baseline
seg_stats = bank.groupby('cluster_label')['y_binary'].agg(['sum', 'count']).reset_index()
seg_stats.columns = ['segment', 'n_yes', 'n_total']
seg_stats['response_rate'] = seg_stats['n_yes'] / seg_stats['n_total']
seg_stats['response_pct'] = (seg_stats['response_rate'] * 100).round(2)
print(seg_stats.to_string(index=False))
print()

# Z-Test: Experienced vs. Mass Market
exp = seg_stats[seg_stats['segment'] == 'Experienced'].iloc[0]
mass = seg_stats[seg_stats['segment'] == 'Mass Market'].iloc[0]

# Gepoolte Proportion
p_pool = (exp['n_yes'] + mass['n_yes']) / (exp['n_total'] + mass['n_total'])
se = np.sqrt(p_pool * (1 - p_pool) * (1/exp['n_total'] + 1/mass['n_total']))
z_stat = (exp['response_rate'] - mass['response_rate']) / se
p_z = 1 - norm.cdf(z_stat)  # einseitig: Experienced > Mass Market

# Effektstaerke: Cohen's h
h = 2 * np.arcsin(np.sqrt(exp['response_rate'])) - 2 * np.arcsin(np.sqrt(mass['response_rate']))

print(f'Z-Test: Experienced ({exp["response_pct"]}%) vs. Mass Market ({mass["response_pct"]}%)')
print(f'  Z-Statistik: {z_stat:.4f}')
print(f'  p-Wert (einseitig): {p_z:.8f}')
print(f"  Cohen's h: {abs(h):.4f}  ({'klein' if abs(h) < 0.2 else 'mittel' if abs(h) < 0.5 else 'gross'})")
print(f'  Lift: {exp["response_rate"]/mass["response_rate"]:.2f}x')
print()
if p_z < ALPHA:
    print(f'  -> Signifikant: Experienced reagiert messbar besser als Mass Market (p={p_z:.2e})')
else:
    print(f'  -> Nicht signifikant (p={p_z:.4f})')

    segment  n_yes  n_total  response_rate  response_pct
Experienced    866     3379       0.256289         25.63
Mass Market   4423    41832       0.105732         10.57

Z-Test: Experienced (25.63%) vs. Mass Market (10.57%)
  Z-Statistik: 26.1925
  p-Wert (einseitig): 0.00000000
  Cohen's h: 0.3993  (mittel)
  Lift: 2.42x

  -> Signifikant: Experienced reagiert messbar besser als Mass Market (p=0.00e+00)


## 4. H1 -- Visualisierung: Response-Rate je Segment mit Konfidenzintervallen

In [5]:
# 95%-Konfidenzintervall (Wilson-Methode)
def wilson_ci(n_yes, n_total, alpha=0.05):
    p = n_yes / n_total
    z = norm.ppf(1 - alpha/2)
    denom = 1 + z**2/n_total
    center = (p + z**2/(2*n_total)) / denom
    margin = z * np.sqrt(p*(1-p)/n_total + z**2/(4*n_total**2)) / denom
    return center - margin, center + margin

seg_stats['ci_low'], seg_stats['ci_high'] = zip(*[
    wilson_ci(row['n_yes'], row['n_total']) for _, row in seg_stats.iterrows()
])
seg_stats['ci_low_pct']  = seg_stats['ci_low']  * 100
seg_stats['ci_high_pct'] = seg_stats['ci_high'] * 100
overall_rate = bank['y_binary'].mean() * 100

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2ecc71' if r > overall_rate else '#e74c3c' for r in seg_stats['response_pct']]
bars = ax.bar(seg_stats['segment'], seg_stats['response_pct'], color=colors, width=0.5)

# Konfidenzintervalle
for i, (_, row) in enumerate(seg_stats.iterrows()):
    ax.errorbar(i, row['response_pct'],
                yerr=[[row['response_pct'] - row['ci_low_pct']],
                      [row['ci_high_pct'] - row['response_pct']]],
                fmt='none', color='black', capsize=6, linewidth=2)
    ax.text(i, row['response_pct'] + 1.2, str(row['response_pct']) + '%',
            ha='center', fontweight='bold', fontsize=11)

ax.axhline(y=overall_rate, color='black', linestyle='--', label=f'Gesamt-O ({overall_rate:.1f}%)')
ax.set_title('H1-Test: Response-Rate je Segment mit 95%-Konfidenzintervall', fontsize=12)
ax.set_ylabel('Response-Rate (%)')
ax.set_xlabel('Segment (Bank Marketing)')
ax.legend()

# p-Wert Annotation
ax.annotate(f'Chi2-Test: p={p_value:.2e}\nCramers V={cramers_v:.4f}',
            xy=(0.98, 0.95), xycoords='axes fraction', ha='right', va='top',
            fontsize=9, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('../outputs/hyp_h1_response_ci.png', dpi=150, bbox_inches='tight')
plt.close()
print('Plot gespeichert.')

Plot gespeichert.


## 5. H1 -- Robustheitspruefung: Mehrere demografische Sub-Gruppen

In [6]:
# Erweiterte H1-Pruefung: Top-Job-Segmente aus EDA
# Pruefen ob andere natuerliche Gruppen (job, education) aehnliche Unterschiede zeigen

results = []
for group_col in ['job', 'education', 'marital']:
    ct = pd.crosstab(bank[group_col], bank['y_binary'])
    chi2_g, p_g, dof_g, _ = chi2_contingency(ct)
    n_g = ct.values.sum()
    cv_g = np.sqrt(chi2_g / (n_g * (min(ct.shape) - 1)))
    results.append({'Merkmal': group_col, 'chi2': round(chi2_g, 2),
                    'p_value': round(p_g, 6), 'cramers_v': round(cv_g, 4),
                    'signifikant': 'ja' if p_g < ALPHA else 'nein'})

results_df = pd.DataFrame(results)
print('Erweiterter Chi-Quadrat-Test: Verschiedene Segmentierungsmerkmale')
print(results_df.to_string(index=False))
print()
print('-> Alle Merkmale signifikant: Segmentierung ist robust, nicht nur bei cluster_label')

Erweiterter Chi-Quadrat-Test: Verschiedene Segmentierungsmerkmale
  Merkmal   chi2  p_value  cramers_v signifikant
      job 836.11      0.0     0.1360          ja
education 238.92      0.0     0.0727          ja
  marital 196.50      0.0     0.0659          ja

-> Alle Merkmale signifikant: Segmentierung ist robust, nicht nur bei cluster_label


## 6. H2 -- Silhouette-Score: Statistische Guete der Segmentierung

**Nullhypothese (H0):** Die Cluster sind zufaellig -- kein Unterschied zu Random-Baseline.

**Test:** Permutationstest -- Vergleich des echten Silhouette-Scores mit N zufaelligen Cluster-Zuordnungen.

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# GA4 RFM Features
RFM = ['recency_days', 'n_sessions', 'n_events', 'n_view_item', 'total_revenue']
X = ga4[RFM].copy()
for col in RFM:
    X[col] = X[col].clip(upper=X[col].quantile(0.99))
X_scaled = StandardScaler().fit_transform(X)

# Echter Silhouette-Score (Tier-1)
km2 = KMeans(n_clusters=2, random_state=42, n_init=20)
true_labels = km2.fit_predict(X_scaled)
true_sil = silhouette_score(X_scaled, true_labels, sample_size=5000, random_state=42)
print(f'Echter Silhouette-Score (k=2): {true_sil:.4f}')

# Permutationstest: N=100 Random-Shuffle
N_PERM = 100
np.random.seed(42)
perm_scores = []
for i in range(N_PERM):
    random_labels = np.random.permutation(true_labels)
    s = silhouette_score(X_scaled, random_labels, sample_size=2000, random_state=i)
    perm_scores.append(s)

perm_scores = np.array(perm_scores)
p_perm = (perm_scores >= true_sil).mean()

print(f'Permutationstest (N={N_PERM}):')
print(f'  Echter Score:         {true_sil:.4f}')
print(f'  Random-Baseline (O):  {perm_scores.mean():.4f} (SD={perm_scores.std():.4f})')
print(f'  p-Wert (permutiert):  {p_perm:.4f}')
print()
if p_perm < ALPHA:
    print(f'  -> H0 ABGELEHNT: Cluster sind signifikant besser als Zufall (p={p_perm:.4f})')
    print('  -> H2 GESTATETZT: Segmentierung ist statistisch valide')
else:
    print(f'  -> H0 NICHT ABGELEHNT (p={p_perm:.4f})')

Echter Silhouette-Score (k=2): 0.7175


Permutationstest (N=100):
  Echter Score:         0.7175
  Random-Baseline (O):  -0.0113 (SD=0.0533)
  p-Wert (permutiert):  0.0000

  -> H0 ABGELEHNT: Cluster sind signifikant besser als Zufall (p=0.0000)
  -> H2 GESTATETZT: Segmentierung ist statistisch valide


In [8]:
# Visualisierung Permutationstest
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(perm_scores, bins=20, color='steelblue', alpha=0.7, label='Random-Baseline (N=100)')
ax.axvline(x=true_sil, color='red', linewidth=2.5, label=f'Echter Score ({true_sil:.4f})')
ax.axvline(x=np.percentile(perm_scores, 95), color='orange', linestyle='--',
           label=f'95%-Perzentil Baseline ({np.percentile(perm_scores, 95):.4f})')
ax.set_title('H2-Test: Silhouette-Score vs. Permutations-Baseline', fontsize=12)
ax.set_xlabel('Silhouette-Score')
ax.set_ylabel('Haeufigkeit')
ax.legend()
ax.annotate(f'p={p_perm:.4f}', xy=(true_sil, ax.get_ylim()[1]*0.85),
            ha='left', fontsize=11, color='red', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/hyp_h2_silhouette_permtest.png', dpi=150, bbox_inches='tight')
plt.close()
print('Plot gespeichert.')

Plot gespeichert.


## 7. H2 -- Conversion-Rate-Vergleich je GA4-Segment

In [9]:
# H2: Treffsicherheit -- zeigt sich in Conversion-Rate-Unterschieden zwischen Segmenten
ga4_stats = ga4.groupby('cluster_label')['has_purchased'].agg(['sum', 'count']).reset_index()
ga4_stats.columns = ['segment', 'n_converted', 'n_total']
ga4_stats['conv_rate'] = ga4_stats['n_converted'] / ga4_stats['n_total']
ga4_stats['conv_pct'] = (ga4_stats['conv_rate'] * 100).round(3)
overall_conv = ga4['has_purchased'].mean() * 100

print('GA4 Conversion-Rate je Segment:')
print(ga4_stats.to_string(index=False))
print(f'\nGesamt-Baseline: {overall_conv:.3f}%')

# Champions vs. Passive: Chi-Quadrat
champ = ga4_stats[ga4_stats['segment'] == 'Champions'].iloc[0]
passive = ga4_stats[ga4_stats['segment'] == 'Passive'].iloc[0]

ct_ga4 = np.array([[champ['n_converted'], champ['n_total'] - champ['n_converted']],
                    [passive['n_converted'], passive['n_total'] - passive['n_converted']]])
chi2_ga4, p_ga4, _, _ = chi2_contingency(ct_ga4)

lift_champ = champ['conv_rate'] / passive['conv_rate'] if passive['conv_rate'] > 0 else float('inf')
print(f'\nChi2-Test Champions vs. Passive:')
print(f'  Champions: {champ["conv_pct"]}% | Passive: {passive["conv_pct"]}%')
print(f'  chi2={chi2_ga4:.2f} | p={p_ga4:.6f}')
print(f'  Lift: {lift_champ:.1f}x')
print(f'  -> {"Signifikant" if p_ga4 < ALPHA else "Nicht signifikant"} (alpha={ALPHA})')

GA4 Conversion-Rate je Segment:
     segment  n_converted  n_total  conv_rate  conv_pct
   Champions           52      491   0.105906    10.591
Loyal Buyers          131     3463   0.037828     3.783
     Passive           90    63865   0.001409     0.141

Gesamt-Baseline: 0.403%

Chi2-Test Champions vs. Passive:
  Champions: 10.591% | Passive: 0.141%
  chi2=2369.47 | p=0.000000
  Lift: 75.2x
  -> Signifikant (alpha=0.05)


In [10]:
# Visualisierung: Conversion-Rate je GA4-Segment
ga4_stats_sorted = ga4_stats.sort_values('conv_pct', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bar_cols = ['#2ecc71' if r > overall_conv else '#e74c3c' for r in ga4_stats_sorted['conv_pct']]
bars = axes[0].bar(ga4_stats_sorted['segment'], ga4_stats_sorted['conv_pct'], color=bar_cols)
axes[0].axhline(y=overall_conv, color='black', linestyle='--',
                label=f'Gesamt-O ({overall_conv:.3f}%)')
axes[0].set_title('H2: Conversion-Rate je GA4-Segment', fontsize=12)
axes[0].set_ylabel('Conversion-Rate (%)')
axes[0].legend()
for bar, val in zip(bars, ga4_stats_sorted['conv_pct']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 str(val)+'%', ha='center', fontweight='bold', fontsize=10)

# Lift-Plot
ga4_stats_sorted['lift'] = (ga4_stats_sorted['conv_rate'] / (overall_conv/100)).round(2)
lift_cols = ['#2ecc71' if l >= 1 else '#e74c3c' for l in ga4_stats_sorted['lift']]
axes[1].bar(ga4_stats_sorted['segment'], ga4_stats_sorted['lift'], color=lift_cols)
axes[1].axhline(y=1.0, color='black', linestyle='--', label='Baseline (1.0x)')
axes[1].set_title('H2: Lift je Segment vs. Gesamt-Baseline', fontsize=12)
axes[1].set_ylabel('Lift (x)')
axes[1].legend()
for i, (_, row) in enumerate(ga4_stats_sorted.iterrows()):
    axes[1].text(i, row['lift']+0.2, str(row['lift'])+'x', ha='center', fontweight='bold')

plt.suptitle('GA4 Segmentierung -- Conversion-Rate & Lift (H2)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/hyp_h2_conversion_lift.png', dpi=150, bbox_inches='tight')
plt.close()
print('Plot gespeichert.')

Plot gespeichert.


## 8. Zusammenfassung: H1 & H2 Ergebnisse

In [11]:
print('=' * 70)
print('HYPOTHESENTEST-ERGEBNISSE')
print('=' * 70)

print()
print('H1: Personalisierte Ansprache erzielt signifikant hoehere Response-Rate')
print('-' * 70)
h1_result = 'BESTAETIGT' if p_value < ALPHA else 'NICHT BESTAETIGT'
print(f'  Ergebnis:       {h1_result}')
print(f'  Test:           Chi-Quadrat + Z-Test (Proportionen)')
print(f'  Chi2:           {chi2:.2f} | df={dof} | p={p_value:.2e}')
print(f'  Cramers V:      {cramers_v:.4f} (Effektstaerke)')
print(f'  Experienced:    {exp["response_pct"]}% vs. Mass Market: {mass["response_pct"]}%')
print(f'  Z-Test p:       {p_z:.2e} (einseitig)')
print(f'  Lift:           {exp["response_rate"]/mass["response_rate"]:.2f}x')

print()
print('H2: Segmentierung verbessert Treffsicherheit messbar')
print('-' * 70)
h2_result = 'BESTAETIGT' if p_perm < ALPHA else 'NICHT BESTAETIGT'
print(f'  Ergebnis:       {h2_result}')
print(f'  Silhouette:     {true_sil:.4f} vs. Random-Baseline O={perm_scores.mean():.4f}')
print(f'  Permtest p:     {p_perm:.4f} (N=100 Permutationen)')
print(f'  Champions Conv: {champ["conv_pct"]}% vs. Passive: {passive["conv_pct"]}%')
print(f'  Lift Champions: {lift_champ:.1f}x')

print()
print('TABELLE:')
results_tbl = pd.DataFrame([
    {'Hypothese': 'H1', 'Ergebnis': h1_result, 'Test': 'Chi2', 'p-Wert': round(p_value, 6),
     'Effektstaerke': f'Cramers V={cramers_v:.3f}', 'Lift': str(round(exp["response_rate"]/mass["response_rate"], 2)) + 'x'},
    {'Hypothese': 'H2', 'Ergebnis': h2_result, 'Test': 'Permutationstest',
     'p-Wert': round(p_perm, 4), 'Effektstaerke': f'Sil={true_sil:.4f}',
     'Lift': str(round(lift_champ, 1)) + 'x (Champions)'},
])
print(results_tbl.to_string(index=False))

# Als CSV speichern fuer Bericht
results_tbl.to_csv('../outputs/hypothesis_results.csv', index=False)
print('\nGespeichert: outputs/hypothesis_results.csv')

HYPOTHESENTEST-ERGEBNISSE

H1: Personalisierte Ansprache erzielt signifikant hoehere Response-Rate
----------------------------------------------------------------------
  Ergebnis:       BESTAETIGT
  Test:           Chi-Quadrat + Z-Test (Proportionen)
  Chi2:           684.59 | df=1 | p=6.71e-151
  Cramers V:      0.1231 (Effektstaerke)
  Experienced:    25.63% vs. Mass Market: 10.57%
  Z-Test p:       0.00e+00 (einseitig)
  Lift:           2.42x

H2: Segmentierung verbessert Treffsicherheit messbar
----------------------------------------------------------------------
  Ergebnis:       BESTAETIGT
  Silhouette:     0.7175 vs. Random-Baseline O=-0.0113
  Permtest p:     0.0000 (N=100 Permutationen)
  Champions Conv: 10.591% vs. Passive: 0.141%
  Lift Champions: 75.2x

TABELLE:
Hypothese   Ergebnis             Test  p-Wert   Effektstaerke              Lift
       H1 BESTAETIGT             Chi2     0.0 Cramers V=0.123             2.42x
       H2 BESTAETIGT Permutationstest     0.0      S

In [12]:
# Summary-Visual: Beide Hypothesen auf einem Bild
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# H1: Segment-Vergleich Bank
h1_cols = ['#2ecc71' if r > overall_rate else '#e74c3c' for r in seg_stats['response_pct']]
bars1 = axes[0].bar(seg_stats['segment'], seg_stats['response_pct'], color=h1_cols, width=0.5)
axes[0].axhline(y=overall_rate, color='black', linestyle='--', label=f'Baseline ({overall_rate:.1f}%)')
for bar, row in zip(bars1, seg_stats.itertuples()):
    axes[0].errorbar(bar.get_x()+bar.get_width()/2, row.response_pct,
                     yerr=[[row.response_pct - row.ci_low_pct],[row.ci_high_pct - row.response_pct]],
                     fmt='none', color='black', capsize=5)
    axes[0].text(bar.get_x()+bar.get_width()/2, row.response_pct+1.0,
                 str(row.response_pct)+'%', ha='center', fontweight='bold')
axes[0].set_title(f'H1: {h1_result}\n(Chi2 p={p_value:.2e}, Lift={round(exp["response_rate"]/mass["response_rate"],2)}x)', fontsize=11)
axes[0].set_ylabel('Response-Rate (%)')
axes[0].legend(fontsize=9)

# H2: Silhouette Permutationstest
axes[1].hist(perm_scores, bins=20, color='steelblue', alpha=0.7, label='Random-Baseline')
axes[1].axvline(x=true_sil, color='red', linewidth=2.5, label=f'Echter Score ({true_sil:.4f})')
axes[1].axvline(x=np.percentile(perm_scores, 95), color='orange', linestyle='--',
                label=f'95%-Quantil')
axes[1].set_title(f'H2: {h2_result}\n(Permtest p={p_perm:.4f}, Sil={true_sil:.4f})', fontsize=11)
axes[1].set_xlabel('Silhouette-Score')
axes[1].set_ylabel('Haeufigkeit')
axes[1].legend(fontsize=9)

plt.suptitle('Hypothesentests H1 + H2 -- Ergebnisübersicht', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/hyp_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print('Summary-Plot gespeichert.')

Summary-Plot gespeichert.


## 9. Interpretation fuer den Bericht

### H1 -- Conversion/Response
- **Befund:** Der Chi-Quadrat-Test zeigt einen hochsignifikanten Zusammenhang zwischen Segment-Zugehoerigkeit und Response-Rate (p << 0.05).
- **Effektstaerke:** Cramers V zeigt schwachen bis mittleren Effekt -- die Segmentierung erklaert einen messbaren, aber nicht dominanten Anteil der Varianz.
- **Praktische Relevanz:** Experienced-Segment erzielt 2.2x hoehere Response als Mass Market. Bei 3.379 Kunden dieses Segments bedeutet gezieltes Targeting eine deutlich effizientere Kampagne.

### H2 -- Segmentierungsqualitaet
- **Befund:** Silhouette-Score (>0.70) ist hochsignifikant besser als zufaellige Cluster (Permtest p=0.00).
- **Praktische Relevanz:** Champions-Segment zeigt massiv hoehere Conversion als Passive -- Segmentierung trennt tatsaechlich relevante Verhaltensunterschiede.

### Limitation
- GA4 nur Nov-Dez 2020: saisonale Verzerrung moeglich (Weihnachtsgeschaeft).
- Bank Marketing: duration-Variable (Data Leakage) wurde aus Analyse ausgeschlossen.
- Kausale Schluesse nicht moeglich -- Korrelationsstudie.